In [1]:
import pandas as pd

In [2]:
# let's grab our data
file = r"C:\Users\bkuhn\Downloads\Out of Service Counts - July 9, 2026.xlsx"
df = pd.read_excel(file)
df.head()

,Year-Week Number,20' OOS Units,40/45' OOS Units,OOS Units
0,2026-26,503.0,1710.0,2213.0
1,2026-25,503.0,1710.0,2213.0
2,2026-24,558.0,1874.0,2432.0
3,2026-23,617.0,1361.0,1978.0
4,2026-22,433.0,1627.0,2060.0


In [3]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

# 1. Chronological order (file is newest-first) --------------------------------
d = df.copy()
yw = d["Year-Week Number"].astype(str).str.extract(r"(\d{4})-(\d{1,2})")

bad = d.loc[yw[0].isna(), "Year-Week Number"]
if len(bad):
    print(f"Dropping {len(bad)} unparseable row(s): {bad.tolist()}")

d = (d.assign(_yr=yw[0], _wk=yw[1])
       .dropna(subset=["_yr", "_wk"])
       .astype({"_yr": int, "_wk": int})
       .sort_values(["_yr", "_wk"])
       .reset_index(drop=True))

cols = ["20' OOS Units", "40/45' OOS Units", "OOS Units"]

def assess(series: pd.Series, label: str, train_frac: float = 0.7):
    y = series.to_numpy(dtype=float)
    prev, nxt = y[:-1], y[1:]                 # (value_{t-1}, value_t) pairs
    n = len(nxt)

    full = sm.OLS(nxt, sm.add_constant(prev)).fit()   # lag-1 autocorr + significance
    r = np.sign(full.params[1]) * np.sqrt(full.rsquared)

    # time-ordered one-step-ahead backtest, fit on history only
    k = max(2, int(n * train_frac))
    coef = sm.OLS(nxt[:k], sm.add_constant(prev[:k])).fit()
    te_prev, te_nxt = prev[k:], nxt[k:]

    mae = lambda a, b: float(np.mean(np.abs(a - b)))
    mae_ar   = mae(te_nxt, coef.predict(sm.add_constant(te_prev, has_constant="add")))
    mae_last = mae(te_nxt, te_prev)           # persistence
    mae_mean = mae(te_nxt, nxt[:k].mean())    # guess the average

    print(f"\n{label}  (n={n} transitions, test={len(te_nxt)})")
    print(f"  lag-1 autocorr r={r:+.2f}  p={full.pvalues[1]:.3f}  slope={coef.params[1]:+.2f}")
    print(f"  out-of-sample MAE  history={mae_ar:7.1f}  persistence={mae_last:7.1f}  mean={mae_mean:7.1f}")
    print(f"  -> history {'IS' if mae_ar < mae_mean else 'is NOT'} predictive "
          f"(vs mean baseline: {(mae_mean - mae_ar) / mae_mean:+.0%})")

for c in cols:
    assess(d[c], c)

Dropping 1 unparseable row(s): ['-']

20' OOS Units  (n=424 transitions, test=128)
  lag-1 autocorr r=+0.97  p=0.000  slope=+0.98
  out-of-sample MAE  history=   85.8  persistence=   86.8  mean=  341.1
  -> history IS predictive (vs mean baseline: +75%)

40/45' OOS Units  (n=424 transitions, test=128)
  lag-1 autocorr r=+0.96  p=0.000  slope=+0.97
  out-of-sample MAE  history=  286.6  persistence=  288.0  mean= 1179.1
  -> history IS predictive (vs mean baseline: +76%)

OOS Units  (n=424 transitions, test=128)
  lag-1 autocorr r=+0.97  p=0.000  slope=+0.98
  out-of-sample MAE  history=  309.4  persistence=  310.5  mean= 1497.8
  -> history IS predictive (vs mean baseline: +79%)


In [4]:
from statsmodels.tsa.stattools import adfuller
for c in cols:
    p = adfuller(d[c].dropna())[1]
    print(f"{c}: ADF p={p:.3f} -> {'stationary' if p < 0.05 else 'unit root (random-walk-like)'}")

20' OOS Units: ADF p=0.149 -> unit root (random-walk-like)
40/45' OOS Units: ADF p=0.099 -> unit root (random-walk-like)
OOS Units: ADF p=0.495 -> unit root (random-walk-like)


In [5]:
from statsmodels.stats.diagnostic import acorr_ljungbox

def assess_changes(series, label, train_frac=0.7):
    dy = series.diff().dropna().to_numpy(dtype=float)   # week-over-week changes
    n = len(dy)

    adf_p = adfuller(dy)[1]                              # are the changes stationary?

    drift = sm.OLS(dy, np.ones(n)).fit()                # is mean change != 0?
    mean_change, drift_p = float(drift.params[0]), float(drift.pvalues[0])

    lb_p = float(acorr_ljungbox(dy, lags=[4])["lb_pvalue"].iloc[-1])  # white noise?

    # one-step backtest on the CHANGE scale ------------------------------------
    prev, nxt = dy[:-1], dy[1:]                          # (change_{t-1}, change_t)
    m = len(nxt)
    k = max(2, int(m * train_frac))
    ar = sm.OLS(nxt[:k], sm.add_constant(prev[:k])).fit()   # AR(1) on changes
    te_prev, te_nxt = prev[k:], nxt[k:]

    mae = lambda a, b: float(np.mean(np.abs(a - b)))
    mae_ar    = mae(te_nxt, ar.predict(sm.add_constant(te_prev, has_constant="add")))
    mae_drift = mae(te_nxt, dy[:k].mean())              # forecast change = training mean
    mae_zero  = mae(te_nxt, 0.0)                        # forecast no change = level persistence

    print(f"\n{label}  (n={n} changes, test={len(te_nxt)})")
    print(f"  changes stationary? ADF p={adf_p:.3f} -> {'yes' if adf_p < 0.05 else 'no'}")
    print(f"  mean change={mean_change:+.1f}/wk  drift p={drift_p:.3f} -> {'drift' if drift_p < 0.05 else 'no drift'}")
    print(f"  autocorr in changes? Ljung-Box p={lb_p:.3f} -> {'structure' if lb_p < 0.05 else 'white noise'}")
    print(f"  out-of-sample MAE  AR(1)={mae_ar:7.1f}  drift={mae_drift:7.1f}  no-change={mae_zero:7.1f}")
    best = min([('AR(1)', mae_ar), ('drift', mae_drift), ('no-change', mae_zero)], key=lambda x: x[1])
    print(f"  -> best: {best[0]}")

for c in cols:
    assess_changes(d[c], c)


20' OOS Units  (n=424 changes, test=127)
  changes stationary? ADF p=0.000 -> yes
  mean change=+1.1/wk  drift p=0.834 -> no drift
  autocorr in changes? Ljung-Box p=0.003 -> structure
  out-of-sample MAE  AR(1)=   85.9  drift=   87.5  no-change=   87.4
  -> best: AR(1)

40/45' OOS Units  (n=424 changes, test=127)
  changes stationary? ADF p=0.000 -> yes
  mean change=+3.7/wk  drift p=0.793 -> no drift
  autocorr in changes? Ljung-Box p=0.006 -> structure
  out-of-sample MAE  AR(1)=  285.5  drift=  287.5  no-change=  287.2
  -> best: AR(1)

OOS Units  (n=424 changes, test=127)
  changes stationary? ADF p=0.000 -> yes
  mean change=+4.7/wk  drift p=0.769 -> no drift
  autocorr in changes? Ljung-Box p=0.003 -> structure
  out-of-sample MAE  AR(1)=  305.5  drift=  310.7  no-change=  309.8
  -> best: AR(1)
